In [102]:
import pandas as pd
import numpy as np
from alerce.core import Alerce
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord

alerce = Alerce()

In [103]:
tns = pd.read_csv("tns_search.csv")
column_names = tns.columns.tolist()
print(column_names)

ztf = pd.read_csv("ZTF_new.csv")
column_names = ztf.columns.tolist()
print(column_names)


['ID', 'Name', 'RA', 'DEC', 'Obj. Type', 'Redshift', 'Host Name', 'Host Redshift', 'Reporting Group/s', 'Discovery Data Source/s', 'Classifying Group/s', 'Associated Group/s', 'Disc. Internal Name', 'Disc. Instrument/s', 'Class. Instrument/s', 'TNS AT', 'Public', 'End Prop. Period', 'Discovery Mag/Flux', 'Discovery Filter', 'Discovery Date (UT)', 'Sender', 'Remarks', 'Unreal', 'Discovery Bibcode', 'Classification Bibcodes', 'Ext. catalog/s', 'Auto classification']
['oid', 'ndethist', 'ncovhist', 'mjdstarthist', 'mjdendhist', 'corrected', 'stellar', 'ndet', 'g_r_max', 'g_r_max_corr', 'g_r_mean', 'g_r_mean_corr', 'firstmjd', 'lastmjd', 'deltajd', 'meanra', 'meandec', 'sigmara', 'sigmadec', 'class', 'classifier', 'probability', 'step_id_corr']


In [104]:
def convert_ra(ra_coords):
    ra_new = SkyCoord(ra_coords,)
    
    return ra_new


In [105]:
def convert_dec(dec_coords):
    dec_coords = str(dec_coords)
    
    degrees, mins, secs = dec_coords.split(":")

    if float(degrees) < 0:
        dec_new = (float(degrees) - float(mins) / 60 - float(mins) / 3600)
        
    else:
        dec_new = (float(degrees) + float(mins) / 60 + float(secs) / 3600)

    return dec_new


In [106]:
def convert_mjd(times):
    t = Time(times, format='iso', scale='utc')
    
    return t.mjd


In [107]:
def convert_coords(tns_ra_coords, tns_dec_coords):
    new_vals = SkyCoord(tns_ra_coords, tns_dec_coords, unit = (u.hourangle, u.deg))

    return new_vals


In [108]:
df_tns_coords = convert_coords(tns['RA'].values, tns['DEC'].values)

tns_ra = df_tns_coords.ra
tns_dec = df_tns_coords.dec

ztf_ra = ztf['meanra'].values
ztf_dec = ztf['meandec'].values

tns_coords = SkyCoord(ra=tns_ra,dec=tns_dec)
ztf_coords = SkyCoord(ra=ztf_ra*u.degree, dec=ztf_dec*u.degree)

idx, d2d, d3d = tns_coords.match_to_catalog_sky(ztf_coords) # Look at TNS and find the closest ZTF
# idx = index of closest ZTF object to each TNS object 
# d2d = angular distance in the sky between the TNS object and the closest ZTF object




In [ ]:
max_sep = 1.0 * u.arcsec
sep_constraint = d2d < max_sep
matches = tns_coords[sep_constraint]

tns['match_index'] = idx
tns['distance'] = d2d

tns_matched_df = tns.iloc[sep_constraint].copy()

ztf_sub = ztf[ztf.index.isin(tns_matched_df['match_index'].values)]

ztf2tns = ztf_sub.merge(tns_matched_df,
                        left_index = True,
                        right_on = 'match_index',
                        how = 'inner'
)

ztf2tns.to_csv('ztf_tns_matches_20arcsec.csv', index=False)

In [110]:
max_sep = 5.0 * u.arcsec
sep_constraint = d2d < max_sep
matches = tns_coords[sep_constraint]

tns['match_index'] = idx
tns['distance'] = d2d

tns_matched_df = tns.iloc[sep_constraint].copy()

ztf_sub = ztf[ztf.index.isin(tns_matched_df['match_index'].values)]

ztf2tns = ztf_sub.merge(tns_matched_df,
                        left_index = True,
                        right_on = 'match_index',
                        how = 'inner'
)

ztf2tns.to_csv('ztf_tns_matches_5arcsec.csv', index=False)

In [111]:
max_sep = 10.0 * u.arcsec
sep_constraint = d2d < max_sep
matches = tns_coords[sep_constraint]

tns['match_index'] = idx
tns['distance'] = d2d

tns_matched_df = tns.iloc[sep_constraint].copy()

ztf_sub = ztf[ztf.index.isin(tns_matched_df['match_index'].values)]

ztf2tns = ztf_sub.merge(tns_matched_df,
                        left_index = True,
                        right_on = 'match_index',
                        how = 'inner'
)

ztf2tns.to_csv('ztf_tns_matches_10arcsec.csv', index=False)

In [112]:
max_sep = 20.0 * u.arcsec
sep_constraint = d2d < max_sep
matches = tns_coords[sep_constraint]

tns['match_index'] = idx
tns['distance'] = d2d

tns_matched_df = tns.iloc[sep_constraint].copy()

ztf_sub = ztf[ztf.index.isin(tns_matched_df['match_index'].values)]

ztf2tns = ztf_sub.merge(tns_matched_df,
                        left_index = True,
                        right_on = 'match_index',
                        how = 'inner'
)

ztf2tns.to_csv('ztf_tns_matches_20arcsec.csv', index=False)

In [113]:
print(np.sort(d2d.arcsec)[:20])

[7.28448439e-02 1.30025886e-01 1.42678892e-01 1.66064917e-01
 3.11489455e-01 1.11324215e+01 1.31870311e+01 1.96176580e+01
 5.40517588e+01 1.44559383e+02 2.21468356e+02 2.30945506e+02
 2.35875190e+02 2.70590537e+02 2.85582986e+02 2.86934966e+02
 2.94827341e+02 3.28026158e+02 3.62076122e+02 3.70480192e+02]


In [114]:
print("Min:", d2d.arcsec.min())
print("Med:", np.median(d2d.arcsec))
print("Max:", d2d.arcsec.max())

Min: 0.07284484392525363
Med: 2457.4169993132255
Max: 157416.11771551045


In [115]:
print(tns_matched_df.shape)

(8, 30)


In [116]:
print("Number of TNS objects:", len(tns))
print("Number of ZTF objects:", len(ztf))

print("Matches within 20 arcsec:", sep_constraint.sum())

print("\n20 smallest separations:")
print(np.sort(d2d.arcsec)[:20])

print("\nZTF index:")
print(ztf.index[:10])

Number of TNS objects: 313
Number of ZTF objects: 35950
Matches within 20 arcsec: 8

20 smallest separations:
[7.28448439e-02 1.30025886e-01 1.42678892e-01 1.66064917e-01
 3.11489455e-01 1.11324215e+01 1.31870311e+01 1.96176580e+01
 5.40517588e+01 1.44559383e+02 2.21468356e+02 2.30945506e+02
 2.35875190e+02 2.70590537e+02 2.85582986e+02 2.86934966e+02
 2.94827341e+02 3.28026158e+02 3.62076122e+02 3.70480192e+02]

ZTF index:
RangeIndex(start=0, stop=10, step=1)
